In [1]:
!pip install fair-esm torch numpy

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import torch
import esm
import numpy as np

In [3]:
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

batch_converter = alphabet.get_batch_converter()

model.eval()

ESM2(
  (embed_tokens): Embedding(33, 1280, padding_idx=1)
  (layers): ModuleList(
    (0-32): 33 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=1280, out_features=5120, bias=True)
      (fc2): Linear(in_features=5120, out_features=1280, bias=True)
      (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=660, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (emb_layer_norm_after): LayerNorm((1280,), eps=1

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Device:", device)

Device: cuda


In [5]:
pdl1_sequence = (
"MFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYNKINQRILVVDPVTSEHELTCQAEGYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTNEIFYCTFRRLDPEENHTAELVIPELPLAHPPNERT"
)

In [6]:
data = [("PDL1", pdl1_sequence)]

labels, strs, tokens = batch_converter(data)

tokens = tokens.to(device)

print("Token shape:", tokens.shape)

Token shape: torch.Size([1, 224])


In [7]:
with torch.no_grad():

    outputs = model(tokens, repr_layers=[33])

representations = outputs["representations"][33]

In [8]:
sequence_length = len(pdl1_sequence)

residue_embeddings = representations[0, 1:sequence_length+1]

print("Residue embedding shape:", residue_embeddings.shape)

Residue embedding shape: torch.Size([222, 1280])


In [9]:
protein_embedding = residue_embeddings.mean(0)

protein_embedding = protein_embedding.cpu().numpy()

print("Protein embedding shape:", protein_embedding.shape)

Protein embedding shape: (1280,)


In [10]:
np.save("pdl1_embedding.npy", protein_embedding)

In [11]:
protein_embedding[:20]

array([ 0.01953585, -0.02638784, -0.0933927 ,  0.06063402, -0.1390264 ,
       -0.03372013,  0.09583247,  0.02027446, -0.09474574,  0.16312295,
        0.1211248 , -0.02663608,  0.0648922 ,  0.01451186,  0.0562801 ,
       -0.02865061,  0.12383723,  0.12610811, -0.02318446,  0.07398442],
      dtype=float32)